In [1]:
# ============================================================
# WHAT IS A TRANSFORM?
# ============================================================
# Right now your images are all different sizes (some 1000x800, some 500x400)
# The AI needs ALL images to be the SAME size — like how all passport photos
# must be the same size!
#
# We also do "augmentation" = making fake variations of images
# e.g., flipping an X-ray horizontally = a new training image for free!
# ============================================================

import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt

# --- TRAINING transforms (with augmentation = extra fake images) ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),          # Step 1: Resize to 224x224 pixels
    transforms.Grayscale(num_output_channels=3),  # Step 2: Convert to 3-channel (models expect this)
    transforms.RandomHorizontalFlip(p=0.5), # Step 3: 50% chance flip image (augmentation)
    transforms.RandomRotation(degrees=10),  # Step 4: Randomly rotate up to 10 degrees
    transforms.ColorJitter(brightness=0.2), # Step 5: Slightly vary brightness
    transforms.ToTensor(),                  # Step 6: Convert image → numbers (0.0 to 1.0)
    transforms.Normalize(                   # Step 7: Normalize (center the numbers around 0)
        mean=[0.485, 0.456, 0.406],         #   These magic numbers come from ImageNet dataset
        std=[0.229, 0.224, 0.225]           #   We use them because our pretrained model was
    )                                       #   trained on ImageNet — so we match its expectations
])

# --- VALIDATION & TEST transforms (NO augmentation — we test on real images only) ---
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✅ Transforms defined!")
print("\nTraining pipeline:")
print("  Image → Resize 224x224 → Flip? → Rotate? → Brightness? → Numbers → Normalize")
print("\nVal/Test pipeline:")
print("  Image → Resize 224x224 → Numbers → Normalize")

✅ Transforms defined!

Training pipeline:
  Image → Resize 224x224 → Flip? → Rotate? → Brightness? → Numbers → Normalize

Val/Test pipeline:
  Image → Resize 224x224 → Numbers → Normalize


In [2]:
# ============================================================
# ImageFolder is a magic PyTorch tool
# It looks at your folder structure and automatically knows:
#   NORMAL    → label 0
#   PNEUMONIA → label 1
# You don't have to label anything manually!
# ============================================================

base_dir = r"C:\Users\SS\OneDrive\Desktop\University\Semester 6\pneumonia_project\chest_xray"

train_dataset = datasets.ImageFolder(base_dir + r"\train", transform=train_transform)
val_dataset   = datasets.ImageFolder(base_dir + r"\val",   transform=val_test_transform)
test_dataset  = datasets.ImageFolder(base_dir + r"\test",  transform=val_test_transform)

print("Classes found:", train_dataset.classes)
print("Class → Number mapping:", train_dataset.class_to_idx)
print(f"\nTraining samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples   : {len(test_dataset)}")

Classes found: ['NORMAL', 'PNEUMONIA']
Class → Number mapping: {'NORMAL': 0, 'PNEUMONIA': 1}

Training samples  : 5216
Validation samples: 16
Testing samples   : 624


In [3]:
# ============================================================
# FIXING THE IMBALANCE
# ============================================================
# Idea: Give NORMAL images a higher "lottery ticket chance"
# of being picked during training.
#
# If NORMAL is 26% of data → give it weight = 1/0.26 = 3.8
# If PNEUMONIA is 74% → give it weight = 1/0.74 = 1.35
# Now both classes get picked roughly equally often!
# ============================================================

# Count images per class
targets = train_dataset.targets   # list like [0,0,1,1,1,0,1,...]
class_counts = np.bincount(targets)
print(f"Class counts → NORMAL: {class_counts[0]}, PNEUMONIA: {class_counts[1]}")

# Calculate weight for each class
class_weights = 1.0 / class_counts
print(f"Class weights → NORMAL: {class_weights[0]:.4f}, PNEUMONIA: {class_weights[1]:.4f}")

# Give every image a weight based on which class it belongs to
sample_weights = [class_weights[t] for t in targets]
sample_weights = torch.DoubleTensor(sample_weights)

# The sampler picks images with these weights (NORMAL gets picked more often)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\n✅ Sampler created! NORMAL images will now be picked more often during training.")

Class counts → NORMAL: 1341, PNEUMONIA: 3875
Class weights → NORMAL: 0.0007, PNEUMONIA: 0.0003

✅ Sampler created! NORMAL images will now be picked more often during training.


In [4]:
# ============================================================
# WHAT IS A DATALOADER?
# ============================================================
# Imagine a factory conveyor belt.
# You can't feed 5000 images to the AI all at once (not enough memory!)
# So DataLoader sends them in small batches — like 32 images at a time.
#
# batch_size=32 → AI sees 32 images, learns, then sees next 32, etc.
# num_workers=0 → how many helpers load images (0 = main process, safe on Windows)
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,        # use our weighted sampler (fixes imbalance)
    num_workers=0,
    pin_memory=True         # faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # no shuffling for val/test
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✅ DataLoaders ready!")
print(f"Training batches  : {len(train_loader)}  (each batch = {BATCH_SIZE} images)")
print(f"Validation batches: {len(val_loader)}")
print(f"Testing batches   : {len(test_loader)}")

✅ DataLoaders ready!
Training batches  : 163  (each batch = 32 images)
Validation batches: 1
Testing batches   : 20


In [5]:
# ============================================================
# Let's grab ONE batch and inspect it
# This is like tasting the food before serving it 👨‍🍳
# ============================================================

images, labels = next(iter(train_loader))

print(f"One batch shape : {images.shape}")
print(f"  → means: {images.shape[0]} images, {images.shape[1]} channels, {images.shape[2]}x{images.shape[3]} pixels")
print(f"Labels in batch : {labels.tolist()}")
print(f"  → 0=NORMAL, 1=PNEUMONIA")
print(f"Pixel value range after normalize: {images.min():.2f} to {images.max():.2f}")
print(f"\n✅ Data pipeline is working! Ready to train the AI.")

# Count 0s and 1s in this batch to see if balancing worked
zeros = (labels == 0).sum().item()
ones  = (labels == 1).sum().item()
print(f"\nIn this batch → NORMAL: {zeros}, PNEUMONIA: {ones}  (should be roughly equal now!)")

One batch shape : torch.Size([32, 3, 224, 224])
  → means: 32 images, 3 channels, 224x224 pixels
Labels in batch : [1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0]
  → 0=NORMAL, 1=PNEUMONIA
Pixel value range after normalize: -2.12 to 2.64

✅ Data pipeline is working! Ready to train the AI.

In this batch → NORMAL: 16, PNEUMONIA: 16  (should be roughly equal now!)
